# My SmartQ Data Understanding & Exploratory Data Analysis (EDA)

I use this notebook for the **Data Understanding** stage of my SmartQ CRISP-DM workflow.

I treat EDA as the practical part of Data Understanding, but I know Data Understanding is slightly broader. I use this stage to check data quality, understand my prediction target, identify useful variables, inspect missing values and outliers, and find fields that could cause data leakage.

My prediction target is:

`actual_wait_minutes`

My dataset contains 100,000 synthetic SmartQ operational queue records.

I do this stage first because I do not want to trust model results before I understand the data I am giving the models.


## 1. I load the dataset

I wrote the code below so I can run the notebook either from the repository root or from the `notebooks/` folder.

I want the notebook to be reproducible instead of depending on one exact working directory.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_NAME = "SmartQ_Synthetic_Operational_Dataset_100k.csv"

candidate_paths = [
    Path("data") / DATASET_NAME,
    Path("..") / "data" / DATASET_NAME,
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Run this notebook from the repository root "
        "or from the notebooks folder."
    )

df = pd.read_csv(data_path)

print("Dataset:", data_path.resolve())
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


## 2. I parse timestamps and define my modelling population

For waiting-time regression, I use **completed visits only**.

I exclude cancelled and no-show records because those records do not contain a genuine completed waiting-time outcome.

I keep them in the master dataset because they can still be useful for other SmartQ analysis later.


In [ ]:
datetime_columns = [
    "scenario_date",
    "appointment_at",
    "arrival_at",
    "check_in_at",
    "service_eligible_at",
    "call_time",
    "service_started_at",
    "service_completed_at",
]

for column in datetime_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

completed = df[df["status"] == "COMPLETED"].copy()

print(f"All records: {len(df):,}")
print(f"Completed records available for regression: {len(completed):,}")
print(f"Excluded no-shows/cancellations: {len(df) - len(completed):,}")


## 3. I inspect the basic structure and data types

I check the shape, column types and descriptive statistics before doing deeper analysis.

I do this because wrong data types or unexpected values can create modelling problems later.


In [ ]:
display(df.info())
display(df.describe(include="all").T)


## 4. I inspect status, service, branch and queue composition

I use these counts to understand what kinds of records exist in my dataset and whether the different operational groups are represented.


In [ ]:
for column in [
    "status",
    "branch_name",
    "service_name",
    "booking_source",
    "queue_type",
    "day_of_week",
    "is_peak_period",
]:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).to_frame("count"))


## 5. I analyse missing values

I do not assume every missing value is bad data.

I expect some missing values by design:

- `appointment_at` is blank for walk-ins because they do not have appointments.
- recent-history features can be blank early in the day because there may not yet be enough previous completed/called customers.
- outcome fields are blank for no-shows and cancellations.

My goal is to understand **why** a value is missing before I decide how to handle it.


In [ ]:
missing_all = (
    df.isna()
      .sum()
      .to_frame("missing_rows")
      .assign(missing_pct=lambda x: (x["missing_rows"] / len(df) * 100).round(2))
      .query("missing_rows > 0")
      .sort_values("missing_rows", ascending=False)
)
display(missing_all)

print("\nMissing values inside completed visits only:")
missing_completed = (
    completed.isna()
             .sum()
             .to_frame("missing_rows")
             .assign(missing_pct=lambda x: (x["missing_rows"] / len(completed) * 100).round(2))
             .query("missing_rows > 0")
             .sort_values("missing_rows", ascending=False)
)
display(missing_completed)


## 6. I run data-quality checks

I use these checks to confirm that the synthetic records still obey basic SmartQ queue logic.

I check for impossible waiting times, broken timestamp order, invalid queue positions and service-eligibility problems.

I do this because a model can learn bad data just as easily as good data.


In [ ]:
appointment_rows = completed[completed["booking_source"] == "APPOINTMENT"]
walkin_rows = completed[completed["booking_source"] == "WALK_IN"]

quality_checks = pd.Series({
    "negative_wait_rows": int((completed["actual_wait_minutes"] < 0).sum()),
    "nonpositive_service_rows": int((completed["actual_service_minutes"] <= 0).sum()),
    "queue_position_mismatch": int(
        (completed["queue_position"] != completed["people_ahead"] + 1).sum()
    ),
    "call_before_service_eligibility": int(
        (completed["call_time"] < completed["service_eligible_at"]).sum()
    ),
    "service_start_before_call": int(
        (completed["service_started_at"] < completed["call_time"]).sum()
    ),
    "completion_before_service_start": int(
        (completed["service_completed_at"] < completed["service_started_at"]).sum()
    ),
    "appointment_eligible_before_booked_time": int(
        (appointment_rows["service_eligible_at"] < appointment_rows["appointment_at"]).sum()
    ),
    "walkin_eligibility_not_equal_checkin": int(
        (walkin_rows["service_eligible_at"] != walkin_rows["check_in_at"]).sum()
    ),
}, name="violations")

display(quality_checks.to_frame())


## 7. I understand my target: actual waiting time

I inspect the distribution of `actual_wait_minutes` because this is the number I am trying to predict.

I look at the mean, median, percentiles and histogram so I can see whether the target is balanced, skewed or affected by extreme congestion cases.


In [ ]:
wait_stats = completed["actual_wait_minutes"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)
display(wait_stats.to_frame("actual_wait_minutes"))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(completed["actual_wait_minutes"], bins=60)
ax.set_title("Distribution of Actual SmartQ Waiting Time")
ax.set_xlabel("Actual wait (minutes)")
ax.set_ylabel("Customers")
plt.show()


## 8. I compare waiting time across service, branch, queue lane and booking source

I use these group summaries to see whether important SmartQ operational groups behave differently before I start modelling.


In [ ]:
group_columns = [
    "service_name",
    "branch_name",
    "queue_type",
    "booking_source",
    "is_peak_period",
]

for column in group_columns:
    summary = (
        completed.groupby(column)["actual_wait_minutes"]
                 .agg(["count", "mean", "median"])
                 .round(2)
                 .sort_values("mean", ascending=False)
    )
    print(f"\n--- Wait by {column} ---")
    display(summary)


In [ ]:
branch_wait = (
    completed.groupby("branch_name")["actual_wait_minutes"]
             .mean()
             .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 5))
branch_wait.plot(kind="barh", ax=ax)
ax.set_title("Average Waiting Time by Branch")
ax.set_xlabel("Average wait (minutes)")
ax.set_ylabel("Branch")
plt.tight_layout()
plt.show()


## 9. I inspect time-of-day and weekday patterns

I check whether waiting time changes by hour and weekday.

I keep these patterns in mind when deciding whether time variables could help the models.


In [ ]:
hourly_wait = completed.groupby("hour_of_day")["actual_wait_minutes"].mean()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weekday_wait = (
    completed.groupby("day_of_week")["actual_wait_minutes"]
             .mean()
             .reindex(weekday_order)
)

display(hourly_wait.round(2).to_frame("avg_wait_minutes"))
display(weekday_wait.round(2).to_frame("avg_wait_minutes"))

fig, ax = plt.subplots(figsize=(9, 5))
hourly_wait.plot(marker="o", ax=ax)
ax.set_title("Average Waiting Time by Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average wait (minutes)")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
weekday_wait.plot(kind="bar", ax=ax)
ax.set_title("Average Waiting Time by Weekday")
ax.set_xlabel("Day")
ax.set_ylabel("Average wait (minutes)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. I compare queue conditions with waiting time

I use this section to check whether queue-state variables move in a sensible direction with waiting time.

I want to know whether features such as people ahead, workload and queue pressure actually contain useful signal before I rely on them in a model.


In [ ]:
numeric_features = [
    "people_ahead",
    "general_waiting",
    "priority_waiting",
    "serving_count",
    "effective_open_counters",
    "counter_utilisation",
    "queue_pressure_index",
    "workload_minutes_ahead",
    "recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10",
    "recent_throughput_60m",
    "service_target_minutes",
    "hour_of_day",
    "actual_wait_minutes",
]

correlations = completed[numeric_features].corr(numeric_only=True)["actual_wait_minutes"]
correlations = correlations.drop("actual_wait_minutes").sort_values(ascending=False)

display(correlations.to_frame("correlation_with_actual_wait"))


In [ ]:
sample = completed.sample(n=min(5000, len(completed)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sample["queue_pressure_index"],
    sample["actual_wait_minutes"],
    alpha=0.20,
    s=12,
)
ax.set_title("Queue Pressure vs Actual Waiting Time")
ax.set_xlabel("Queue pressure index")
ax.set_ylabel("Actual wait (minutes)")
plt.show()


## 11. I keep the deterministic ETA as an engineering benchmark

The existing SmartQ ETA is **not one of my three official ML models**.

I keep it as an extra benchmark because I want to know whether the machine-learning models improve on the queue logic SmartQ already has.

I do not use the deterministic ETA as an official model input because I want the three ML models to learn from the queue conditions themselves.


In [ ]:
baseline_error = completed["actual_wait_minutes"] - completed["baseline_eta_minutes"]

baseline_mae = baseline_error.abs().mean()
baseline_rmse = np.sqrt((baseline_error ** 2).mean())

print(f"Deterministic SmartQ ETA MAE:  {baseline_mae:.2f} minutes")
print(f"Deterministic SmartQ ETA RMSE: {baseline_rmse:.2f} minutes")


## 12. I audit for data leakage

For a prediction made at check-in, I only use information that would actually be available at that moment.

I exclude fields that contain future/outcome information or directly reveal the answer.

I do this because data leakage can make a model look unrealistically accurate while making it useless in a real application.


In [ ]:
target = "actual_wait_minutes"

leakage_columns = [
    "call_time",
    "actual_wait_minutes",
    "wait_variance_minutes",
    "actual_service_minutes",
    "service_variance_minutes",
    "counter_number",
    "service_started_at",
    "service_completed_at",
    "status",
    "no_show",
    "service_within_target",
    "wait_within_30_minutes",
]

display(pd.DataFrame({"excluded_leakage_or_outcome_field": leakage_columns}))


## 13. My initial EDA conclusions

Before I move to modelling, I confirm the following:

- I understand the dataset dimensions and status counts.
- I use completed visits as the regression population.
- I understand expected missing values instead of deleting them blindly.
- I found no impossible negative waits or broken timestamp ordering in my defined checks.
- waiting time changes with queue conditions and some branch/time patterns.
- my target is right-skewed, so MAE and RMSE both give me useful information.
- I have excluded leakage fields from model inputs.

My next stage is **Data Preparation**, where I choose the final features, handle missing recent-history values, encode categories and build a chronological train/validation/test split.


## 14. I add clearer visual comparisons

I added these charts because I wanted my EDA to answer practical questions instead of only producing generic graphs.

I ask:

- Do General and Priority customers wait differently?
- Do appointments and walk-ins wait differently?
- Are peak periods worse?
- Do service types have different raw average waits?


In [ ]:
comparison_specs = [
    ("queue_type", "Average wait by queue lane", "Queue lane"),
    ("booking_source", "Average wait by booking source", "Booking source"),
    ("is_peak_period", "Average wait: peak vs non-peak", "Peak period"),
    ("service_name", "Average wait by service type", "Service"),
]

for column, title, xlabel in comparison_specs:
    summary = (
        completed.groupby(column)["actual_wait_minutes"]
                 .mean()
                 .sort_values()
    )
    fig, ax = plt.subplots(figsize=(8, 4.5))
    summary.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Average wait (minutes)")
    plt.xticks(rotation=25 if column == "service_name" else 0)
    plt.tight_layout()
    plt.show()


## 15. I visualise missing values

A missing value is not automatically bad data.

I use this chart to see **where** missing values occur so I can decide whether the pattern is expected or whether it suggests a data-quality problem.


In [ ]:
missing_completed = completed.isna().sum()
missing_completed = missing_completed[missing_completed > 0].sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
missing_completed.plot(kind="barh", ax=ax)
ax.set_title("Missing values in completed visits")
ax.set_xlabel("Missing rows")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 16. I inspect a correlation matrix

**Correlation** tells me how strongly two numeric variables move together.

- Values near +1 mean they tend to rise together.
- Values near -1 mean one tends to fall when the other rises.
- Values near 0 mean little linear relationship.

I use correlation for exploration.

I do **not** treat correlation as proof of causation.


In [ ]:
correlation_columns = [
    "actual_wait_minutes",
    "people_ahead",
    "general_waiting",
    "priority_waiting",
    "serving_count",
    "effective_open_counters",
    "counter_utilisation",
    "queue_pressure_index",
    "workload_minutes_ahead",
    "recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10",
    "recent_throughput_60m",
]

corr = completed[correlation_columns].corr()

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(correlation_columns)))
ax.set_xticklabels(correlation_columns, rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(len(correlation_columns)))
ax.set_yticklabels(correlation_columns, fontsize=8)
ax.set_title("Correlation matrix: selected SmartQ queue features")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 17. I compare statistical significance with practical importance

Because my dataset is large, very small differences can become statistically significant.

So I report both:

- a **p-value**, which helps me ask whether a difference is statistically detectable;
- an **effect size**, which helps me understand how large the difference actually is.

For two-group comparisons I use Welch's t-test and Cohen's d.

For the three service types I use one-way ANOVA and eta-squared.


In [ ]:
from scipy import stats

def welch_and_d(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)

    pooled_sd = np.sqrt(
        ((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1))
        / (len(a)+len(b)-2)
    )
    cohens_d = (a.mean() - b.mean()) / pooled_sd
    return t_stat, p_value, cohens_d

comparisons = []

for label, column, a_value, b_value in [
    ("General vs Priority", "queue_type", "GENERAL", "PRIORITY"),
    ("Appointment vs Walk-in", "booking_source", "APPOINTMENT", "WALK_IN"),
    ("Peak vs Non-peak", "is_peak_period", True, False),
]:
    a = completed.loc[completed[column] == a_value, "actual_wait_minutes"]
    b = completed.loc[completed[column] == b_value, "actual_wait_minutes"]

    t_stat, p_value, d = welch_and_d(a, b)

    comparisons.append({
        "comparison": label,
        "mean_a": a.mean(),
        "mean_b": b.mean(),
        "p_value": p_value,
        "cohens_d": d,
    })

display(pd.DataFrame(comparisons))

service_groups = [
    group["actual_wait_minutes"].to_numpy()
    for _, group in completed.groupby("service_name")
]

f_stat, p_value = stats.f_oneway(*service_groups)

grand_mean = completed["actual_wait_minutes"].mean()
ss_between = sum(
    len(group) * (group.mean() - grand_mean)**2
    for group in service_groups
)
ss_total = ((completed["actual_wait_minutes"] - grand_mean)**2).sum()
eta_squared = ss_between / ss_total

print(f"Service-type ANOVA p-value: {p_value:.4f}")
print(f"Service-type eta-squared: {eta_squared:.8f}")


### What I learned from the significance tests

In my generated data I found statistically detectable differences between:

- General and Priority waits;
- appointments and walk-ins;
- peak and non-peak periods.

However, the effect sizes are small.

I also found that the three service types do **not** have a meaningful raw overall difference in waiting time.

This taught me not to use p-values alone to decide whether a feature matters.
